# PV power and the Typical Meteorological Year (live)

Beyond raw radiation, PVGIS can model **PV system power** and synthesise a
**Typical Meteorological Year (TMY)**. This notebook switches on PV modelling
for `seriescalc` and pulls a TMY for the same point.

In [ ]:
import tempfile

import matplotlib.pyplot as plt

from earthlens.earthlens import EarthLens

OUT = tempfile.mkdtemp()

## PV system power with `pvcalculation=1`

The PV knobs ride through to the PVGIS query: `pvcalculation=1` turns on the
model, `peakpower` (kWp), `loss` (%), `angle` (tilt °), and `aspect` (azimuth °,
0 = south) describe the array. The result gains a `P` column — PV power in W.

In [ ]:
pv = EarthLens(
    data_source="pvgis",
    variables=["seriescalc"],
    start="2020-01-01",
    end="2020-12-31",
    point=(45.0, 8.0),
    pvcalculation=1,
    peakpower=1,
    loss=14,
    angle=35,
    aspect=0,
    path=OUT,
).download(progress_bar=False)

print("P" in pv.columns, pv["P"].max(), "W peak")
pv[["time", "P", "G(i)"]].head()

A representative summer day shows the modelled PV power profile.

In [ ]:
day = pv[(pv["time"] >= "2020-06-15") & (pv["time"] < "2020-06-16")]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(day["time"], day["P"], color="tab:orange")
ax.set_ylabel("P  [W]")
ax.set_title("PVGIS PV power — 1 kWp, 35° tilt, south, 15 June 2020")
fig.autofmt_xdate()
plt.show()

## A Typical Meteorological Year

`variables=["tmy"]` returns one synthetic 8760-hour year of meteorological
variables (temperature, humidity, irradiance components, wind, pressure) — no
year window needed.

In [ ]:
tmy = EarthLens(
    data_source="pvgis",
    variables=["tmy"],
    start="2020-01-01",
    end="2020-12-31",
    point=(45.0, 8.0),
    path=OUT,
).download(progress_bar=False)

print(tmy.shape)
tmy[["time", "T2m", "RH", "G(h)"]].head()

## Takeaway

The same `download()` shape gives raw radiation, modelled PV power (with the PV
knobs), or a TMY — selected entirely by `variables=` and the keyword knobs. All
keyless, all returned as tidy hourly `DataFrame`s.